# 10. Dynamic programming

**Goal.** Turn an exponential recursion into a polynomial algorithm by storing the answers to repeated subproblems.

**You will be able to:**
1. Recognise overlapping subproblems and optimal substructure.
2. Write the same solution top down (memoisation) and bottom up (tabulation).
3. Compress a table to the rows you still need.
4. Apply the pattern to sequences, knapsack-style choices and string comparison.

## 1. Memoisation and tabulation

### How it works under the hood
- A problem suits DP when it has **optimal substructure**: the best answer comes from best answers to smaller problems.
- It also needs **overlapping subproblems**: the same smaller problems recur.
- Naive Fibonacci recomputes `fib(k)` exponentially often. The call count is `2 * fib(n + 1) - 1`, about 1.6^n.
- **Memoisation** keeps the recursive shape and caches results by arguments. `functools.cache` stores them in a dict. Cost is the number of distinct states times the work per state.
- **Tabulation** fills a table in dependency order with loops. It avoids recursion depth limits and function-call overhead.
- **Space compression**: when a row depends only on the previous row (or two values), keep only those.
- Python limits recursion to about 1000 frames, so deep memoised recursion needs a loop or a raised limit.

### Production use
- Pricing, resource allocation and route planning with state.
- Sequence alignment in bioinformatics, and parsing (CYK, Viterbi in speech and NLP).
- Caching pure recursive functions in build tools and compilers.

### Trade-offs

| Approach | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Naive recursion | O(depth) | Exponential | Mirrors the maths | Trivial to write | Unusable beyond small n |
| Memoisation (`@cache`) | O(states) plus stack | O(states * work) | Almost the naive code | Computes only reachable states | Recursion limit, dict overhead |
| Tabulation | O(states), or less when compressed | O(states * work), lowest constant | More index work | No recursion, easy to compress | Fills states you may never need |
| Greedy | O(1) to O(n) | Fastest | Short | Great when proven correct | Often wrong without a proof |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import functools
import time

from understanding_python import dp


def timed_ms(fn, *args):
    t0 = time.perf_counter()
    out = fn(*args)
    return out, (time.perf_counter() - t0) * 1e3


print(f"{'n':>3} {'naive calls':>12} {'naive ms':>9} {'memo ms':>8} {'table ms':>9}")
for n in (10, 20, 25, 30):
    dp.fib_memo.cache_clear()
    a, t_naive = timed_ms(dp.fib_naive, n)
    b, t_memo = timed_ms(dp.fib_memo, n)
    c, t_table = timed_ms(dp.fib_table, n)
    assert a == b == c
    print(f"{n:>3} {dp.fib_calls(n):>12,} {t_naive:>9.2f} {t_memo:>8.3f} {t_table:>9.4f}")

print("fib(90) by table:", dp.fib_table(90), "| memo entries held after fib(30):", dp.fib_memo.cache_info().currsize)
print("calls the naive version would need for fib(90):", f"{dp.fib_calls(90):,}")

dp.fib_memo.cache_clear()
try:
    dp.fib_memo(5000)
except RecursionError:
    print("memoised recursion fails at n = 5000: RecursionError")
print("tabulation handles it:", len(str(dp.fib_table(5000))), "digits")

  n  naive calls  naive ms  memo ms  table ms
 10          177      0.02    0.010    0.0034
 20       21,891      1.82    0.008    0.0025
 25      242,785     20.21    0.018    0.0049


 30    2,692,537    224.50    0.025    0.0096
fib(90) by table: 2880067194370816120 | memo entries held after fib(30): 31
calls the naive version would need for fib(90): 9,320,093,220,751,060,617
memoised recursion fails at n = 5000: RecursionError
tabulation handles it: 1045 digits


## 2. Choices under a budget: coins, knapsack and subsequences

### How it works under the hood
- **Coin change** (fewest coins): `best[a] = 1 + min(best[a - c])` over coins `c <= a`. Greedy fails for some coin sets, for example (1, 3, 4) and amount 6.
- **0/1 knapsack**: `dp[c]` is the best value with capacity `c`. Process items one at a time and loop capacity downwards so each item is used once. Cost O(items * capacity) time and O(capacity) space. It is *pseudo-polynomial*: capacity counts in units, not bits.
- **Longest increasing subsequence** (LIS): the O(n^2) table compares each element with all earlier ones. The O(n log n) version keeps the smallest tail of each length and uses `bisect` to place each element.
- A brute-force check on small random inputs is the standard way to test a DP.

### Production use
- Coin change: cash-machine and voucher splitting, price combinations.
- Knapsack: choose features for a sprint, ads for a budget, cargo for a container, cloud instances under a spend cap.
- LIS: version histories, patience sorting, longest chain of compatible jobs.

### Trade-offs

| Problem | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Coin change DP | O(amount) | O(amount * coins) | Short | Exact for any coin set | Slow for huge amounts |
| Greedy coins | O(1) | O(coins) | Shortest | Fast | Wrong for non-canonical sets |
| Knapsack DP | O(capacity) | O(items * capacity) | Compact | Exact | Pseudo-polynomial, NP-hard in general |
| Brute-force subsets | O(items) | O(2^items) | Obvious | Great as a test oracle | Only for about 20 items |
| LIS with `bisect` | O(n) | O(n log n) | Needs the tails idea | Fast on long inputs | Returns length, path needs extra work |

In [3]:
import random

from understanding_python.bench import best_time


def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count, amount = count + amount // c, amount % c
    return count if amount == 0 else -1


print("coins (1, 3, 4), amount 6: DP", dp.min_coins([1, 3, 4], 6), "| greedy", greedy_coins([1, 3, 4], 6))
uk = [1, 2, 5, 10, 20, 50, 100, 200]
print("UK coins in pence, 387p:", dp.min_coins(uk, 387), "| greedy agrees on this canonical set:", greedy_coins(uk, 387))
print("impossible amount:", dp.min_coins([5, 10], 3), "| zero amount:", dp.min_coins([5], 0))

features = [("single sign-on", 8, 90), ("dark mode", 3, 25), ("export to csv", 5, 40), ("audit log", 6, 70),
            ("mobile layout", 9, 85), ("bulk import", 7, 60), ("saved searches", 4, 45)]
capacity = 20
weights = [f[1] for f in features]
values = [f[2] for f in features]
best = dp.knapsack(weights, values, capacity)
print(f"sprint capacity {capacity} points: best value {best} | brute force agrees: {best == dp.knapsack_brute(weights, values, capacity)}")

rng = random.Random(12)
checks = 0
for _ in range(200):
    w = [rng.randint(1, 9) for _ in range(10)]
    v = [rng.randint(1, 30) for _ in range(10)]
    cap = rng.randint(0, 40)
    assert dp.knapsack(w, v, cap) == dp.knapsack_brute(w, v, cap)
    checks += 1
print("random knapsack cases verified against brute force:", checks)

series = [rng.randint(0, 10**6) for _ in range(3000)]
assert dp.lis_length(series) == dp.lis_quadratic(series)
t_fast = best_time(lambda: dp.lis_length(series), repeat=3)
t_slow = best_time(lambda: dp.lis_quadratic(series), repeat=1)
print(f"LIS length {dp.lis_length(series)} of 3000 values: O(n log n) {t_fast * 1e3:.1f} ms | O(n^2) {t_slow * 1e3:.0f} ms")

coins (1, 3, 4), amount 6: DP 2 | greedy 3
UK coins in pence, 387p: 7 | greedy agrees on this canonical set: 7
impossible amount: -1 | zero amount: 0
sprint capacity 20 points: best value 205 | brute force agrees: True


random knapsack cases verified against brute force: 200


LIS length 102 of 3000 values: O(n log n) 0.6 ms | O(n^2) 544 ms


## 3. String DP: edit distance, LCS and word break

### How it works under the hood
- **Edit distance** (Levenshtein): `D[i][j]` is the cheapest way to turn the first `i` characters of `s` into the first `j` of `t`. Take the minimum of delete, insert and substitute. Keep two rows for O(len(t)) space.
- **Longest common subsequence** (LCS): if the characters match, add 1 to the diagonal. Otherwise take the better of skipping a character from either string. Walk the table to recover the sequence.
- **Word break**: `parts[end]` holds a segmentation of the first `end` characters. Check each word that could end there. Bounding by the longest dictionary word keeps it O(n * L).
- All three fill a table whose entries depend on the left, upper and diagonal neighbours.

### Production use
- Spell correction and fuzzy search, deduplicating records with typos.
- Diff tools and version control (`git diff` uses LCS-style algorithms), plagiarism detection.
- Tokenisation of text with no spaces: hashtags, URL slugs, Chinese word segmentation.

### Trade-offs

| Algorithm | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Edit distance, two rows | O(len(t)) | O(len(s) * len(t)) | Compact | Exact similarity score | Quadratic for long strings |
| LCS with full table | O(len(a) * len(b)) | O(len(a) * len(b)) | Moderate | Recovers the actual sequence | Table memory |
| Word break DP | O(n) lists | O(n * L) | Moderate | Finds a segmentation if one exists | Returns the first, not the best |
| Brute-force recursion | O(n) stack | Exponential | Short | Easy to reason about | Unusable on real text |

In [4]:
dictionary = ["receive", "recipe", "believe", "separate", "definitely", "necessary", "occurrence", "embarrass"]
for typo in ("recieve", "seperate", "definately", "ocurrence"):
    ranked = sorted(dictionary, key=lambda w: dp.edit_distance(typo, w))
    print(f"{typo:11} -> {ranked[0]:10} (distance {dp.edit_distance(typo, ranked[0])})")
print("kitten to sitting:", dp.edit_distance("kitten", "sitting"), "| empty to 'abc':", dp.edit_distance("", "abc"), "| equal strings:", dp.edit_distance("same", "same"))

old = ["def total(items):", "    subtotal = 0", "    for x in items:", "        subtotal += x", "    return subtotal"]
new = ["def total(items, tax):", "    subtotal = 0", "    for x in items:", "        subtotal += x", "    return subtotal * tax"]
print("longest common subsequence of ABCBDAB and BDCABA:", repr(dp.lcs("ABCBDAB", "BDCABA")))


def line_diff(a, b):
    """Diff two line lists with the LCS idea, using a line id per distinct line."""
    ids = {line: chr(0x100 + i) for i, line in enumerate(dict.fromkeys(a + b))}
    common = set(dp.lcs("".join(ids[x] for x in a), "".join(ids[x] for x in b)))
    return [("= " if ids[x] in common else "- ") + x for x in a], [("= " if ids[x] in common else "+ ") + x for x in b]


removed, added = line_diff(old, new)
print("diff:")
print(*removed, *added, sep="\n")

words = {"stack", "over", "flow", "stackover", "py", "thon", "python", "data", "base", "database"}
for slug in ("stackoverflow", "pythondatabase", "pythonbase", "pyflowx"):
    print(f"{slug:15} ->", dp.word_break(slug, words))
print("empty text:", dp.word_break("", words), "| empty dictionary:", dp.word_break("abc", set()))

recieve     -> receive    (distance 2)
seperate    -> separate   (distance 1)
definately  -> definitely (distance 1)
ocurrence   -> occurrence (distance 1)
kitten to sitting: 3 | empty to 'abc': 3 | equal strings: 0
longest common subsequence of ABCBDAB and BDCABA: 'BDAB'
diff:
- def total(items):
=     subtotal = 0
=     for x in items:
=         subtotal += x
-     return subtotal
+ def total(items, tax):
=     subtotal = 0
=     for x in items:
=         subtotal += x
+     return subtotal * tax
stackoverflow   -> ['stackover', 'flow']
pythondatabase  -> ['python', 'database']
pythonbase      -> ['python', 'base']
pyflowx         -> None
empty text: [] | empty dictionary: None
